# Reddit Historical Collector

Pulls historical Reddit posts and comments from **Arctic Shift** (free Pushshift mirror).
Covers 2005-present across finance and tech subreddits.

## Output files on GitHub
| File | Contents |
|------|----------|
| `reddit_data/reddit_{YEAR}.csv` | Posts + comments for that year |
| `reddit_data/reddit_index.csv` | Coverage summary |

## Subreddits
- r/wallstreetbets, r/stocks, r/investing, r/technology, r/SecurityAnalysis

---
## 0. Install

In [1]:
# !pip install requests pandas python-dotenv

---
## 1. Configuration

In [2]:
import os, io, time, base64, requests
import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv
load_dotenv()

GITHUB_REPO    = 'annhmartin/dataviz-historical-stocks-AnnetteMartin'
GITHUB_TOKEN   = os.environ.get('GITHUB_TOKEN', None)
REDDIT_PREFIX  = 'reddit_data'

SUBREDDITS = [
    'wallstreetbets', 'stocks', 'investing',
    'technology', 'SecurityAnalysis'
]

START_YEAR  = 2006   # Reddit founded 2005, meaningful volume from 2006
END_YEAR    = datetime.now(timezone.utc).year
DELAY       = 1.0    # Arctic Shift rate limit: be polite
BATCH_SIZE  = 500    # posts per API request

# Arctic Shift base URL
ARCTIC_BASE = 'https://arctic-shift.photon-reddit.com/api'

print('Configuration loaded')
print(f'  Repo       : {GITHUB_REPO}')
print(f'  Token      : {"set" if GITHUB_TOKEN else "NOT SET"}')
print(f'  Subreddits : {SUBREDDITS}')
print(f'  Date range : {START_YEAR} to {END_YEAR}')

Configuration loaded
  Repo       : annhmartin/dataviz-historical-stocks-AnnetteMartin
  Token      : set
  Subreddits : ['wallstreetbets', 'stocks', 'investing', 'technology', 'SecurityAnalysis']
  Date range : 2006 to 2026


---
## 2. GitHub helpers

In [3]:
GITHUB_API = 'https://api.github.com'

def _gh_headers(token):
    return {
        'Authorization': f'Bearer {token}',
        'Accept': 'application/vnd.github+json',
        'X-GitHub-Api-Version': '2022-11-28',
    }

def push_csv(df, path, token, msg=None):
    if msg is None:
        msg = f'Update {path} - {len(df):,} rows'
    buf = io.StringIO()
    df.to_csv(buf, index=False)
    encoded = base64.b64encode(buf.getvalue().encode()).decode()
    url = f'{GITHUB_API}/repos/{GITHUB_REPO}/contents/{path}'
    headers = _gh_headers(token)
    check = requests.get(url, headers=headers, timeout=15)
    sha = check.json().get('sha') if check.status_code == 200 else None
    payload = {'message': msg, 'content': encoded}
    if sha: payload['sha'] = sha
    for attempt in range(3):
        resp = requests.put(url, headers=headers, json=payload, timeout=120)
        if resp.status_code in (200, 201):
            print(f'  Saved {path} ({len(df):,} rows)')
            return True
        if resp.status_code == 409 and attempt < 2:
            time.sleep(3)
            check = requests.get(url, headers=headers, timeout=15)
            sha = check.json().get('sha') if check.status_code == 200 else None
            if sha: payload['sha'] = sha
        else:
            print(f'  FAILED {path}: {resp.status_code}')
            return False

def load_csv(path, token=None):
    url = f'https://raw.githubusercontent.com/{GITHUB_REPO}/main/{path}'
    headers = {'Authorization': f'Bearer {token}'} if token else {}
    resp = requests.get(url, headers=headers, timeout=60)
    if resp.status_code == 404: raise FileNotFoundError(path)
    resp.raise_for_status()
    content = resp.text.strip()
    if not content: return pd.DataFrame()
    return pd.read_csv(io.StringIO(content), low_memory=False)

def file_exists(path, token):
    url = f'{GITHUB_API}/repos/{GITHUB_REPO}/contents/{path}'
    return requests.get(url, headers=_gh_headers(token), timeout=10).status_code == 200

print('GitHub helpers loaded')

GitHub helpers loaded


---
## 3. Arctic Shift API helpers

In [4]:
def fetch_reddit_posts(subreddit, after_ts, before_ts, limit=500):
    """
    Fetch posts from Arctic Shift for a subreddit in a time window.
    Returns list of post dicts.
    """
    url = f'{ARCTIC_BASE}/posts/search'
    params = {
        'subreddit' : subreddit,
        'after'     : int(after_ts),
        'before'    : int(before_ts),
        'limit'     : limit,
        'sort'      : 'asc',
        'fields'    : 'id,author,created_utc,subreddit,title,selftext,score,num_comments,url',
    }
    resp = requests.get(url, params=params, timeout=30)
    if resp.status_code != 200:
        return []
    return resp.json().get('data', [])

def fetch_reddit_month(subreddit, year, month):
    """
    Fetch all posts for a subreddit in a calendar month.
    Paginates using created_utc to get all posts.
    """
    from datetime import datetime, timezone
    import calendar

    start_ts = datetime(year, month, 1, tzinfo=timezone.utc).timestamp()
    if month == 12:
        end_ts = datetime(year+1, 1, 1, tzinfo=timezone.utc).timestamp() - 1
    else:
        end_ts = datetime(year, month+1, 1, tzinfo=timezone.utc).timestamp() - 1

    all_posts = []
    cursor = start_ts

    while cursor < end_ts:
        batch = fetch_reddit_posts(subreddit, cursor, end_ts, BATCH_SIZE)
        if not batch:
            break
        all_posts.extend(batch)
        # Move cursor to just after the last post
        cursor = batch[-1]['created_utc'] + 1
        if len(batch) < BATCH_SIZE:
            break
        time.sleep(DELAY)

    return all_posts

def posts_to_df(posts, subreddit):
    if not posts:
        return pd.DataFrame()
    rows = []
    for p in posts:
        created = p.get('created_utc', 0)
        dt = datetime.fromtimestamp(created, tz=timezone.utc)
        rows.append({
            'id'           : p.get('id'),
            'subreddit'    : subreddit,
            'author'       : p.get('author'),
            'title'        : p.get('title', ''),
            'selftext'     : (p.get('selftext') or '')[:500],
            'score'        : p.get('score', 0),
            'num_comments' : p.get('num_comments', 0),
            'created_utc'  : created,
            'date'         : str(dt.date()),
            'year'         : dt.year,
            'month'        : dt.month,
            'day'          : dt.day,
            'day_of_week'  : dt.strftime('%A'),
        })
    return pd.DataFrame(rows)

# Sanity check
print('Testing Arctic Shift API ...')
from datetime import datetime, timezone
test_start = datetime(2023, 1, 1, tzinfo=timezone.utc).timestamp()
test_end   = datetime(2023, 1, 2, tzinfo=timezone.utc).timestamp()
test = fetch_reddit_posts('stocks', test_start, test_end, 5)
print(f'  Test fetch: {len(test)} posts')
if test:
    print(f'  Sample: {test[0].get("title", "")[:80]}')

Testing Arctic Shift API ...
  Test fetch: 5 posts
  Sample: AM I CRAZY OR DO YOU SEE IT TOO


---
## 4. Full Historical Fetch

> Fetches month by month for each subreddit. Saves one CSV per year.
> Crash-safe — skips already-stored year/subreddit combinations.
> **Estimated time:** 2-4 hours for all subreddits 2006-present.

In [10]:
if GITHUB_TOKEN is None:
    print('GITHUB_TOKEN not set.')
else:
    index_rows = []

    for year in range(START_YEAR, END_YEAR + 1):
        year_path = f'{REDDIT_PREFIX}/reddit_{year}.csv'

        # Load existing data for this year
        try:
            df_existing = load_csv(year_path, GITHUB_TOKEN)
            if not df_existing.empty and 'month' in df_existing.columns:
                done = set(zip(
                    df_existing['subreddit'].tolist(),
                    df_existing['month'].astype(int).tolist()
                ))
            else:
                df_existing = pd.DataFrame()
                done = set()
        except FileNotFoundError:
            df_existing = pd.DataFrame()
            done = set()

        new_frames = [df_existing] if not df_existing.empty else []
        current_month = datetime.now(timezone.utc).month

        any_new = False
        for subreddit in SUBREDDITS:
            for month in range(1, 13):
                if year == END_YEAR and month > current_month:
                    continue
                if (subreddit, month) in done:
                    continue
                any_new = True
                month_name = datetime(year, month, 1).strftime('%b')
                posts = fetch_reddit_month(subreddit, year, month)
                df_m = posts_to_df(posts, subreddit)
                if not df_m.empty:
                    new_frames.append(df_m)
                    print(f'  {year} {month_name} r/{subreddit}: {len(df_m):,} posts')
                else:
                    print(f'  {year} {month_name} r/{subreddit}: 0 posts')
                time.sleep(DELAY)

        if not any_new:
            print(f'  {year}: all stored')
            if not df_existing.empty:
                index_rows.append({'year': year, 'post_count': len(df_existing),
                                   'date_min': df_existing['date'].min(),
                                   'date_max': df_existing['date'].max()})
            continue

        if new_frames:
            df_year = (
                pd.concat(new_frames, ignore_index=True)
                .drop_duplicates(subset='id')
                .sort_values('created_utc')
                .reset_index(drop=True)
            )
            if len(df_year) == 0:
                print(f'  {year}: no posts found across all subreddits — skipping')
                continue
            push_csv(df_year, year_path, GITHUB_TOKEN,
                     f'Reddit {year}: {len(df_year):,} posts')
            index_rows.append({'year': year, 'post_count': len(df_year),
                               'date_min': df_year['date'].min(),
                               'date_max': df_year['date'].max()})

    if index_rows:
        df_idx = pd.DataFrame(index_rows).sort_values('year')
        push_csv(df_idx, f'{REDDIT_PREFIX}/reddit_index.csv', GITHUB_TOKEN,
                 f'Reddit index: {df_idx["post_count"].sum():,} total posts')
        print(f'\nDone! Total posts: {df_idx["post_count"].sum():,}')
        print(df_idx.to_string(index=False))

  2006 Jan r/wallstreetbets: 0 posts
  2006 Feb r/wallstreetbets: 0 posts
  2006 Mar r/wallstreetbets: 0 posts
  2006 Apr r/wallstreetbets: 0 posts
  2006 May r/wallstreetbets: 0 posts
  2006 Jun r/wallstreetbets: 0 posts
  2006 Jul r/wallstreetbets: 0 posts
  2006 Aug r/wallstreetbets: 0 posts
  2006 Sep r/wallstreetbets: 0 posts
  2006 Oct r/wallstreetbets: 0 posts
  2006 Nov r/wallstreetbets: 0 posts
  2006 Dec r/wallstreetbets: 0 posts
  2006 Jan r/stocks: 0 posts
  2006 Feb r/stocks: 0 posts
  2006 Mar r/stocks: 0 posts
  2006 Apr r/stocks: 0 posts
  2006 May r/stocks: 0 posts
  2006 Jun r/stocks: 0 posts
  2006 Jul r/stocks: 0 posts
  2006 Aug r/stocks: 0 posts
  2006 Sep r/stocks: 0 posts
  2006 Oct r/stocks: 0 posts
  2006 Nov r/stocks: 0 posts
  2006 Dec r/stocks: 0 posts
  2006 Jan r/investing: 0 posts
  2006 Feb r/investing: 0 posts
  2006 Mar r/investing: 0 posts
  2006 Apr r/investing: 0 posts
  2006 May r/investing: 0 posts
  2006 Jun r/investing: 0 posts
  2006 Jul r/inv

---
## 5. Incremental Update

In [6]:
if GITHUB_TOKEN is None:
    print('GITHUB_TOKEN not set.')
else:
    current_year  = datetime.now(timezone.utc).year
    current_month = datetime.now(timezone.utc).month
    year_path = f'{REDDIT_PREFIX}/reddit_{current_year}.csv'
    try:
        df_existing = load_csv(year_path, GITHUB_TOKEN)
        done = set(zip(df_existing['subreddit'], df_existing['month'].astype(int)))
    except FileNotFoundError:
        df_existing = pd.DataFrame()
        done = set()
    new_frames = [df_existing] if not df_existing.empty else []
    for subreddit in SUBREDDITS:
        for month in range(1, current_month + 1):
            if month < current_month and (subreddit, month) in done:
                continue
            posts = fetch_reddit_month(subreddit, current_year, month)
            df_m = posts_to_df(posts, subreddit)
            if not df_m.empty:
                new_frames.append(df_m)
                print(f'  {subreddit} {month}: {len(df_m):,} posts')
            time.sleep(DELAY)
    if new_frames:
        df_upd = pd.concat(new_frames, ignore_index=True).drop_duplicates('id').sort_values('created_utc').reset_index(drop=True)
        push_csv(df_upd, year_path, GITHUB_TOKEN, f'Reddit {current_year} update: {len(df_upd):,} posts')
        print(f'Done: {len(df_upd):,} posts for {current_year}')

---
## 6. Verify Coverage

In [7]:
import matplotlib.pyplot as plt

df_idx = load_csv(f'{REDDIT_PREFIX}/reddit_index.csv', GITHUB_TOKEN)
print(df_idx.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 4), facecolor='white')
ax.bar(df_idx['year'], df_idx['post_count'], color='#e74c3c', alpha=0.8, edgecolor='white')
ax.set_title('Reddit Posts per Year (all subreddits combined)', fontweight='bold')
ax.set_ylabel('Posts')
ax.set_facecolor('white')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

FileNotFoundError: reddit_data/reddit_index.csv

In [8]:
# Check what Reddit files are already on GitHub
print('Checking Reddit files on GitHub ...')
stored = []

for year in range(2006, 2027):
    path = f'{REDDIT_PREFIX}/reddit_{year}.csv'
    url  = f'https://raw.githubusercontent.com/{GITHUB_REPO}/main/{path}'
    resp = requests.get(url,
                        headers={'Authorization': f'Bearer {GITHUB_TOKEN}'},
                        timeout=10)
    if resp.status_code == 200:
        lines = max(0, len(resp.text.strip().split('\n')) - 1)
        stored.append({'year': year, 'posts': lines})
        print(f'  {year}: {lines:,} posts')
    import time; time.sleep(0.1)

if stored:
    df_stored = pd.DataFrame(stored)
    print(f'\nTotal: {df_stored["posts"].sum():,} posts across {len(stored)} years')
else:
    print('No Reddit files found yet — Section 4 has not run or nothing was saved')

Checking Reddit files on GitHub ...
No Reddit files found yet — Section 4 has not run or nothing was saved


In [9]:
# Test Arctic Shift API directly
from datetime import datetime, timezone

print('Testing Arctic Shift API ...')

test_cases = [
    ('wallstreetbets', 2021, 1),   # peak WSB period - should have lots
    ('stocks', 2020, 3),            # should have data
    ('investing', 2019, 6),         # should have data
]

for subreddit, year, month in test_cases:
    start_ts = datetime(year, month, 1, tzinfo=timezone.utc).timestamp()
    end_ts   = datetime(year, month, 28, tzinfo=timezone.utc).timestamp()
    
    url = f'{ARCTIC_BASE}/posts/search'
    params = {
        'subreddit' : subreddit,
        'after'     : int(start_ts),
        'before'    : int(end_ts),
        'limit'     : 10,
        'sort'      : 'asc',
        'fields'    : 'id,title,created_utc,score',
    }
    resp = requests.get(url, params=params, timeout=30)
    print(f'\nr/{subreddit} {year}-{month:02d}:')
    print(f'  Status : {resp.status_code}')
    if resp.status_code == 200:
        data = resp.json()
        posts = data.get('data', [])
        print(f'  Posts  : {len(posts)}')
        if posts:
            print(f'  Sample : {posts[0].get("title","")[:80]}')
    else:
        print(f'  Response: {resp.text[:200]}')

Testing Arctic Shift API ...

r/wallstreetbets 2021-01:
  Status : 200
  Posts  : 10
  Sample : 3k - 170k since March (Also, buy LIT!!)

r/stocks 2020-03:
  Status : 200
  Posts  : 10
  Sample : With all the posts on putts and calls ive been seeing Id like to try and figure 

r/investing 2019-06:
  Status : 200
  Posts  : 10
  Sample : Justice Department Is Preparing Antitrust Investigation of Google
